In [2]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os
import numpy as np
from tensorflow.keras.preprocessing import image
from PIL import Image
import math
import random

def augment_images(src_dir, target_dir, num_augmented_images):
    datagen = ImageDataGenerator(
        rotation_range=30,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.2,
        shear_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest'
    )

    img_files = [f for f in os.listdir(src_dir) if f.lower().endswith(('jpg', 'jpeg', 'png'))]
    total_existing = len(img_files)
    generated = 0

    if not img_files:
        print("❌ No valid image files found.")
        return

    print(f"Found {total_existing} images in '{src_dir}'. Augmenting to generate {num_augmented_images} images...")

    if not os.path.exists(target_dir):
        os.makedirs(target_dir)

    random.shuffle(img_files)  # Randomize to avoid overfitting to specific images

    # Calculate how many times we need to augment each image
    per_image_augment = math.ceil(num_augmented_images / total_existing)

    for img_file in img_files:
        img_path = os.path.join(src_dir, img_file)
        try:
            img = image.load_img(img_path, target_size=(224, 224))
        except Exception as e:
            print(f"⚠️ Could not load {img_path}: {e}")
            continue

        x = image.img_to_array(img)
        x = np.expand_dims(x, axis=0)

        i = 0
        for batch in datagen.flow(
            x,
            batch_size=1,
            save_to_dir=target_dir,
            save_prefix='aug',
            save_format='jpeg'
        ):
            i += 1
            generated += 1
            if i >= per_image_augment or generated >= num_augmented_images:
                break

        if generated >= num_augmented_images:
            break

    print(f"✅ Done. Generated {generated} augmented images in '{target_dir}'.")

In [3]:
augment_images(
    src_dir='/mnt/k/ml/clg_ml/domain_specific_classification/train/oral_disorder/hypodontia/',
    target_dir='/mnt/k/ml/clg_ml/domain_specific_classification/train/oral_disorder/hypodontia_augmented',
    num_augmented_images=1500
)

Found 1011 images in '/mnt/k/ml/clg_ml/domain_specific_classification/train/oral_disorder/hypodontia/'. Augmenting to generate 1500 images...
✅ Done. Generated 1500 augmented images in '/mnt/k/ml/clg_ml/domain_specific_classification/train/oral_disorder/hypodontia_augmented'.
